9: Basic Document RAG System

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from transformers import pipeline

# 1. Document Context
document_text = """
Acme Corp Remote Work Policy (2025/2026):
Employees are eligible for full remote work after completing 90 days of continuous employment.
Remote staff are provided an initial home office setup stipend of $750.
Core working hours are strictly required between 10:00 AM and 3:00 PM Eastern Standard Time (EST).
Expense reimbursements for internet and utility costs are capped at $100 per month.
All security compliance protocols must be passed bi-annually.
"""

# 2. Split Document into Chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=150, chunk_overlap=20)
chunks = text_splitter.split_text(document_text)

# 3. Create Vector Database with Embeddings
embedding_function = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_db = Chroma.from_texts(texts=chunks, embedding=embedding_function)

# 4. Initialize Local QA LLM Pipeline
qa_generator = pipeline("text2text-generation", model="google/flan-t5-base")

# 5. RAG Pipeline Query Function
def answer_question_with_rag(query):
    # Retrieve relevant context
    retrieved_docs = vector_db.similarity_search(query, k=2)
    context = "\n".join([doc.page_content for doc in retrieved_docs])
    
    # Construct Context-Augmented Prompt
    augmented_prompt = f"Answer the user question using ONLY the provided context.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:"
    
    # Generate Answer
    result = qa_generator(augmented_prompt, max_length=100)
    return result[0]['generated_text'], context

# Query Example
user_query = "What is the home office setup stipend limit?"
answer, context_used = answer_question_with_rag(user_query)

print(f"QUERY: {user_query}")
print(f"\nRETRIEVED CONTEXT:\n{context_used}")
print(f"\nGENERATED ANSWER:\n{answer}")

Device set to use cuda:0


QUERY: What is the home office setup stipend limit?

RETRIEVED CONTEXT:
Remote staff are provided an initial home office setup stipend of $750.
Expense reimbursements for internet and utility costs are capped at $100 per month.
All security compliance protocols must be passed bi-annually.

GENERATED ANSWER:
$750
